# 02b - LLM Triple Extraction with MinIE-style Annotations (ContraDoc)

Variant of notebook 02 with a richer extraction schema. Predicates are now expressed in their **affirmative, certain** form, and polarity / modality / attribution / quantity are captured as separate fields on each `Triple` rather than baked into the predicate string. This is motivated by the iText2KG trial (see `final-report/itext2kg_trial.tex`) and by MinIE (Gashteovski et al. 2017).

**Why a sibling notebook**
- Keeps notebook 02 (free-form predicate baseline) intact for downstream comparison.
- Output goes to a separate file so 03 / 04 can ingest either variant.

**Inputs**
- `data/processed/ContraDoc/ContraDoc.csv`
- `data/processed/ContraDoc/findability.json` - restricts the YES sample to docs whose evidence and ref sentences are all findable in the document text and not self-referencing (notebook 01 EDA).

**Output:** `data/processed/ContraDoc/triples_minie.jsonl` - one JSON object per document. Each triple now carries `polarity`, `polarity_marker`, `modality`, `modality_marker`, `attribution`, `quantity` in addition to `(s, p, o)`.

**Model:** configured via `settings.llm_model_openai_mini` in `config.py` (the cheap-tier OpenAI model used for this budget-constrained MinIE variant).

**Downstream consumers**
- `03_insert_to_neo4j_ContraDoc.ipynb` (or `03b`) - persists the new fields as `:RELATION` edge properties so RQ1 patterns can filter on polarity flips.
- `06_NLI_ContraDoc.ipynb` - still pairs verbatim `source_text` sentences (unchanged).


In [ ]:
import json
from pathlib import Path

import pandas as pd
from pydantic import BaseModel, Field
from tqdm import tqdm
from typing import Literal

from config import MODEL_PRICING, settings
from utils import init_extraction_llm, resolve_gold_sentence, usage_from_raw

INPUT_PATH = Path("data/processed/ContraDoc/ContraDoc.csv")
BALANCED_PATH = Path("data/processed/ContraDoc/balanced_sample.json")
OUTPUT_PATH = Path("data/processed/ContraDoc/triples_minie.jsonl")
USAGE_PATH = Path("data/processed/ContraDoc/triples_minie_usage.jsonl")


## Structured-output schema

The LLM returns a list of sentences in document order, each with its 1-indexed `sentence_id`, verbatim `source_text`, and a (possibly empty) list of `(s, p, o)` triples extracted from that sentence.

In [ ]:
class Triple(BaseModel):
    s: str = Field(
        description=(
            "Subject - noun phrase, with pronouns and bare definites resolved "
            "to their antecedent using document context."
        )
    )
    p: str = Field(
        description=(
            "Predicate - short verbal phrase. Express the relation in its "
            "AFFIRMATIVE, CERTAIN form. Do NOT include negation words "
            "('not', 'never') or modal/possibility words ('may', 'probably', "
            "'might') in the predicate; report those via the polarity / "
            "modality fields below. Use the surface form of the verb only "
            "(e.g. 'donate', 'born_in', 'manage_to_enter')."
        )
    )
    o: str = Field(
        description=(
            "Object - noun phrase, numeric value (use the placeholder 'Q' if "
            "quantity is annotated below), date, or short complement."
        )
    )

    polarity: Literal["+", "-"] = Field(
        default="+",
        description=(
            "'+' if the relation is asserted positively, '-' if the sentence "
            "negates it ('not', 'never', 'no', 'none', 'nothing')."
        ),
    )
    polarity_marker: str | None = Field(
        default=None,
        description=(
            "The exact negation word from the source sentence when polarity "
            "is '-' (e.g. 'not', 'never'). None when polarity is '+'."
        ),
    )
    modality: Literal["CT", "PS"] = Field(
        default="CT",
        description=(
            "'CT' (certain) by default. 'PS' (possible) if the relation is "
            "hedged with modals ('may', 'might', 'could', 'should', 'would'),"
            " adverbs ('probably', 'possibly', 'maybe', 'likely'), or "
            "infinitive phrases ('is going to', 'plans to', 'intends to')."
        ),
    )
    modality_marker: str | None = Field(
        default=None,
        description=(
            "The exact possibility word/phrase from the source sentence when "
            "modality is 'PS' (e.g. 'probably', 'may'). None when modality "
            "is 'CT'."
        ),
    )
    attribution: str | None = Field(
        default=None,
        description=(
            "Supplier of information when the claim is reported speech, "
            "belief, or attributed via 'according to'. E.g. for 'Pinocchio "
            "believes that Superman was born on Krypton' the triple "
            "(Superman, born_on, Krypton) has attribution='Pinocchio'. None "
            "for direct factual statements."
        ),
    )
    quantity: str | None = Field(
        default=None,
        description=(
            "Original surface form of any cardinal/quantity expression that "
            "has been replaced by the placeholder 'Q' in s or o. "
            "E.g. if o='Q cats' then quantity='9' (or 'all', 'almost 100', "
            "etc.). None if no quantity normalization was applied."
        ),
    )


class SentenceExtraction(BaseModel):
    sentence_id: int = Field(description="1-indexed sentence position within the document, in original order.")
    source_text: str = Field(
        description="Verbatim sentence text, exactly as it appears in the document. Do not paraphrase, normalize, or trim."
    )
    triples: list[Triple] = Field(
        description="All claim triples extracted from this sentence. Empty list if the sentence has no extractable claim."
    )
    is_evidence: bool = Field(
        default=False,
        description="True on the ONE sentence matching the Evidence metadata (YES docs only). False for all other sentences and for every sentence in NO docs.",
    )
    ref_index: int | None = Field(
        default=None,
        description="0-based index into the Reference sentences list, set on the sentence matching that Reference (YES docs only). None otherwise.",
    )


class DocumentExtraction(BaseModel):
    sentences: list[SentenceExtraction]


## LLM client and prompt

Model and API key both pulled from `config.settings`. Change `llm_model` in `config.py` to swap models without touching this notebook.

In [ ]:
SYSTEM_PROMPT = """You are an information extractor. Given a document, split it into sentences and extract all claim triples (subject, predicate, object) per sentence, along with semantic annotations on each triple.

Rules for the core triple:
- Resolve pronouns and bare definites to their antecedents using surrounding context (e.g., 'she' -> 'Mrs. Tittlemouse'; 'the company' -> 'Microsoft Israel').
- Subjects and objects should be noun phrases, numeric values, or dates - concise but specific.
- Express the predicate in its AFFIRMATIVE, CERTAIN form. Do NOT bake polarity, modality, or attribution into the predicate string. For example, from "Faust did not make a deal with the Devil", extract p='make_a_deal_with' with polarity='-' and polarity_marker='not'. From "Superman may have been born on Krypton", extract p='born_on' with modality='PS' and modality_marker='may'.

Rules for semantic annotations (per triple):
- polarity: '+' for affirmed claims, '-' if the sentence negates them. polarity_marker holds the exact negation word ('not', 'never', 'no'); None when '+'.
- modality: 'CT' (certain) by default. 'PS' (possible) for hedged claims with modals ('may', 'might', 'could'), adverbs ('probably', 'possibly', 'likely', 'maybe'), or infinitive verbs ('is going to', 'plans to', 'intends to'). modality_marker holds the exact word/phrase; None when 'CT'.
- attribution: when a claim is REPORTED through a believer/sayer ('X believes that Y', 'according to X, Y', 'X said Y'), set attribution=X and put the inner claim Y as the triple. For direct facts, attribution=None.
- quantity: when the subject or object contains a cardinal/quantity expression ('9 cats', 'all cats', 'almost 100 cats'), replace it with the placeholder 'Q' in s/o and put the original surface form in quantity ('9', 'all', 'almost 100'). Skip when there is no quantity to normalize.

Document-structure rules (unchanged from naive extraction):
- A sentence with no extractable claim has an empty `triples` list, but the sentence MUST still appear in the output (with its source_text) so sentence_id stays aligned with the document.
- Preserve sentences in document order. Do not split, merge, paraphrase, or omit them.
- `source_text` must match the document character-for-character (including punctuation and casing).

Rules for contradiction metadata (only when the user message includes an Evidence/Reference block):
- Set `is_evidence=true` on EXACTLY ONE output sentence - the one matching the provided Evidence text. All others MUST have `is_evidence=false`.
- For each Reference [i] in the provided list, set `ref_index=i` on the ONE output sentence matching that Reference text. All other sentences MUST have `ref_index=null`.
- Sentences tagged `is_evidence=true` or `ref_index=i` MUST each carry at least one triple - these are the semantically important sentences.
- If the user message does NOT include a contradiction metadata block, every sentence must have `is_evidence=false` and `ref_index=null`.
"""


# Active extraction model, set in config.py / overridable via .env.
LLM_MODEL = settings.llm_model_extraction
llm = init_extraction_llm(
    LLM_MODEL,
    openai_key=settings.openai_api_key,
    anthropic_key=settings.anthropic_api_key,
    temperature=0,
)
# include_raw=True so we can read usage_metadata off the AIMessage and bill each call.
extractor = llm.with_structured_output(DocumentExtraction, include_raw=True)


def extract_document(
    text: str,
    evidence: str | None = None,
    refs: list[str] | None = None,
) -> tuple[DocumentExtraction, dict]:
    """Returns (parsed extraction, usage dict). usage dict keys: model,
    input_tokens, output_tokens, total_tokens, *_token_details, input_cost,
    output_cost, total_cost."""
    user_msg = f"Document:\n{text}"
    if evidence is not None:
        user_msg += (
            "\n\n--- Contradiction metadata ---"
            "\nEvidence sentence (tag the matching output sentence with is_evidence=true; extract at least one triple from it):"
            f"\n{evidence}"
        )
    if refs:
        user_msg += (
            "\n\nReference sentence(s) that the Evidence contradicts "
            "(tag each matching output sentence with the correct ref_index; extract at least one triple from each):"
        )
        for i, ref in enumerate(refs):
            user_msg += f"\n  [{i}] {ref}"

    out = extractor.invoke(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ]
    )
    return out["parsed"], usage_from_raw(out["raw"], LLM_MODEL)


## Load ContraDoc

In [ ]:
contra_df = pd.read_csv(INPUT_PATH)
print(f"Loaded {len(contra_df)} documents from {INPUT_PATH}")
print(f"  YES: {(contra_df['contradiction'] == 'YES').sum()}  NO: {(contra_df['contradiction'] == 'NO').sum()}")

balanced = json.loads(BALANCED_PATH.read_text(encoding="utf-8"))
balanced_ids = set(balanced["doc_ids"])
print(f"Balanced sample: {len(balanced_ids)} YES doc_ids (notebook 01)")

# Restrict to the canonical balanced YES sample. No NO docs - the negative
# signal for downstream NLI comes from in-doc non-gold sentence pairings.
contra_df = contra_df[contra_df["id"].astype(str).isin(balanced_ids)].reset_index(drop=True)
print(f"To extract: {len(contra_df)} docs")
contra_df.head(2)


## Sanity check on a single document

Run extraction on the first document to confirm the prompt and schema behave before launching the full pass.

In [ ]:
sample = contra_df[contra_df["contradiction"] == "YES"].iloc[0]
refs = sample["ref_sentences"].split("|")
result, usage = extract_document(sample["text"], evidence=sample["evidence"], refs=refs)

print(f"doc_id={sample['id']}  contradiction={sample['contradiction']}  model={LLM_MODEL}")
print(f"sentences extracted: {len(result.sentences)}")
print(f"total triples:      {sum(len(s.triples) for s in result.sentences)}")
print(f"tokens (in/out/total): {usage['input_tokens']:,} / {usage['output_tokens']:,} / {usage['total_tokens']:,}")
print(f"cost (in/out/total):   ${usage['input_cost']:.4f} / ${usage['output_cost']:.4f} / ${usage['total_cost']:.4f}")

ev = next((s for s in result.sentences if s.is_evidence), None)
ref_ids_by_idx = {s.ref_index: s.sentence_id for s in result.sentences if s.ref_index is not None}
print(f"tagged is_evidence  sentence_id: {ev.sentence_id if ev else None}")
print(f"tagged ref sentence_ids by index: {ref_ids_by_idx}")
print()


def fmt_triple(t: Triple) -> str:
    """Compact one-liner including non-default annotations."""
    extras = []
    if t.polarity == "-":
        extras.append(f"polarity=-:{t.polarity_marker}")
    if t.modality == "PS":
        extras.append(f"modality=PS:{t.modality_marker}")
    if t.attribution is not None:
        extras.append(f"attribution={t.attribution}")
    if t.quantity is not None:
        extras.append(f"quantity={t.quantity}")
    suffix = f"  [{', '.join(extras)}]" if extras else ""
    return f"({t.s}, {t.p}, {t.o}){suffix}"


for s in result.sentences[:8]:
    tag = "EVIDENCE" if s.is_evidence else (f"REF[{s.ref_index}]" if s.ref_index is not None else "        ")
    print(f"[{s.sentence_id}] {tag}  {s.source_text[:120]}")
    for t in s.triples:
        print(f"            -> {fmt_triple(t)}")

print()
print("=== Gold pair triples (with annotations) ===")
if ev is not None:
    print(f"EVIDENCE [{ev.sentence_id}]: {ev.source_text}")
    for t in ev.triples:
        print(f"  -> {fmt_triple(t)}")
for sid in sorted(ref_ids_by_idx.values()):
    s = next(x for x in result.sentences if x.sentence_id == sid)
    print(f"REF      [{s.sentence_id}]: {s.source_text}")
    for t in s.triples:
        print(f"  -> {fmt_triple(t)}")


In [ ]:
result.sentences

In [ ]:
sample["evidence"]

In [ ]:
sample["contra_type"]

In [ ]:
sample

## Run extraction on the full corpus

Resumable: reads any existing `triples.jsonl` and skips documents already done. Output is appended one JSON object per document, flushed every row, so an interrupted run can resume mid-corpus.

Failed extractions are logged and skipped; rerunning the cell will retry them.

In [ ]:
# Re-run this cell repeatedly to extract one chunk at a time. Done docs are
# skipped via the existing OUTPUT_PATH resume. Set CHUNK_SIZE = None to run
# everything remaining in a single pass.
CHUNK_SIZE: int | None = 50

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

done_ids: set[str] = set()
if OUTPUT_PATH.exists():
    with OUTPUT_PATH.open(encoding="utf-8") as f:
        for line in f:
            done_ids.add(json.loads(line)["doc_id"])

# Replay the usage file to show cumulative spend before this chunk starts.
cum_cost = 0.0
cum_in = 0
cum_out = 0
if USAGE_PATH.exists():
    with USAGE_PATH.open(encoding="utf-8") as uf:
        for line in uf:
            u = json.loads(line)
            cum_cost += u.get("total_cost", 0.0)
            cum_in += u.get("input_tokens", 0)
            cum_out += u.get("output_tokens", 0)

print(f"Already extracted: {len(done_ids)} docs")
print(f"Cumulative spend:  ${cum_cost:.4f}  ({cum_in:,} in + {cum_out:,} out)")

remaining = contra_df[~contra_df["id"].astype(str).isin(done_ids)].reset_index(drop=True)
this_chunk = remaining if CHUNK_SIZE is None else remaining.head(CHUNK_SIZE)
print(f"This chunk:        {len(this_chunk)} docs (of {len(remaining)} remaining)  model={LLM_MODEL}")
print()

chunk_cost = 0.0
chunk_in = 0
chunk_out = 0

with OUTPUT_PATH.open("a", encoding="utf-8") as f, USAGE_PATH.open("a", encoding="utf-8") as uf:
    for row in tqdm(this_chunk.itertuples(index=False), total=len(this_chunk)):
        try:
            if row.contradiction == "YES":
                refs = row.ref_sentences.split("|")
                result, usage = extract_document(row.text, evidence=row.evidence, refs=refs)
            else:
                refs = []
                result, usage = extract_document(row.text)
        except Exception as exc:
            print(f"FAILED doc_id={row.id}: {type(exc).__name__}: {exc}")
            continue

        gold_evidence_id = None
        gold_ref_ids: list[int] = []
        resolution_notes: list[str] = []

        if row.contradiction == "YES":
            ev_tagged = next((s for s in result.sentences if s.is_evidence), None)
            gold_evidence_id, ev_res = resolve_gold_sentence(row.evidence, ev_tagged, result.sentences)
            if ev_res != "llm_tag":
                resolution_notes.append(f"evidence={ev_res}")

            ref_by_index = {s.ref_index: s for s in result.sentences if s.ref_index is not None}
            for i, ref_text in enumerate(refs):
                sid, ref_res = resolve_gold_sentence(ref_text, ref_by_index.get(i), result.sentences)
                if sid is not None and sid not in gold_ref_ids:
                    gold_ref_ids.append(sid)
                if ref_res != "llm_tag":
                    resolution_notes.append(f"ref[{i}]={ref_res}")

            if resolution_notes:
                print(f"INFO doc_id={row.id}: {', '.join(resolution_notes)}")
            if gold_evidence_id is None:
                print(f"WARN doc_id={row.id}: evidence unmatched (not in LLM output even by fuzzy)")
            if not gold_ref_ids:
                print(f"WARN doc_id={row.id}: no refs matched")

        record = {
            "doc_id": str(row.id),
            "contradiction": row.contradiction,
            "doc_type": row.doc_type,
            "scope": row.scope,
            "contra_plug": row.contra_plug,
            "contra_type": row.contra_type,
            "evidence": row.evidence,
            "ref_sentences": row.ref_sentences,
            "gold_evidence_sentence_id": gold_evidence_id,
            "gold_ref_sentence_ids": gold_ref_ids,
            "sentences": [s.model_dump() for s in result.sentences],
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        f.flush()

        usage_record = {"doc_id": str(row.id), "contradiction": row.contradiction, **usage}
        uf.write(json.dumps(usage_record, ensure_ascii=False) + "\n")
        uf.flush()

        chunk_cost += usage["total_cost"]
        chunk_in += usage["input_tokens"]
        chunk_out += usage["output_tokens"]

print()
print(f"Chunk done:        ${chunk_cost:.4f}  ({chunk_in:,} in + {chunk_out:,} out)")
print(f"Cumulative spend:  ${cum_cost + chunk_cost:.4f}")
print(f"Still remaining:   {len(remaining) - len(this_chunk)} docs")
print(f"Output: {OUTPUT_PATH.resolve()}")
print(f"Usage:  {USAGE_PATH.resolve()}")


## Verify output

In [ ]:
records = [json.loads(line) for line in OUTPUT_PATH.open(encoding="utf-8")]
n_sentences = sum(len(r["sentences"]) for r in records)
n_triples = sum(sum(len(s["triples"]) for s in r["sentences"]) for r in records)

print(f"Documents extracted: {len(records)} / {len(contra_df)}")
print(f"Total sentences:    {n_sentences}")
print(f"Total triples:      {n_triples}")
print(f"Avg triples / doc:  {n_triples / max(len(records), 1):.1f}")
print()


def triples_at(rec, sid):
    if sid is None:
        return []
    for s in rec["sentences"]:
        if s["sentence_id"] == sid:
            return s["triples"]
    return []


yes_records = [r for r in records if r["contradiction"] == "YES"]
gold_matched = [r for r in yes_records if r["gold_evidence_sentence_id"] is not None and r["gold_ref_sentence_ids"]]
gold_usable = [
    r for r in gold_matched if triples_at(r, r["gold_evidence_sentence_id"]) and any(triples_at(r, sid) for sid in r["gold_ref_sentence_ids"])
]

print(f"YES docs: {len(yes_records)}")
print(f"  with both evidence + ref sentence_ids matched: {len(gold_matched)}")
print(f"  AND >= 1 triple at evidence and at >= 1 ref:   {len(gold_usable)}")
print()

if gold_usable:
    r = gold_usable[0]
    ev_t = triples_at(r, r["gold_evidence_sentence_id"])
    ref_t = [t for sid in r["gold_ref_sentence_ids"] for t in triples_at(r, sid)]
    print(f"Sample usable gold pair (doc_id={r['doc_id']}):")
    print(f"  evidence triples ({len(ev_t)}):")
    for t in ev_t:
        print(f"    {t}")
    print(f"  ref triples ({len(ref_t)}):")
    for t in ref_t:
        print(f"    {t}")